# 01 · Extract
**Purpose:** Pull raw cryptocurrency market data from the [CoinGecko public API](https://www.coingecko.com/en/api/documentation).  
No API key required. Free tier allows up to 30 calls/min.

---
### What this notebook covers
1. Install dependencies  
2. Define constants and configure logging  
3. `fetch_market_data()` — per-coin prices, volume, market cap  
4. `fetch_global_stats()` — total market cap, BTC/ETH dominance  
5. Inspect raw API responses  


In [ ]:
# Install dependencies (Codespaces / first run)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "-q"])
print("✓ requests ready")


In [ ]:
import requests
import logging
import json
from datetime import datetime
from typing import Optional

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

BASE_URL = "https://api.coingecko.com/api/v3"

# Coins to track — edit freely
COINS = [
    "bitcoin", "ethereum", "solana", "cardano",
    "polkadot", "chainlink", "avalanche-2", "uniswap"
]

print(f"Tracking {len(COINS)} coins: {', '.join(COINS)}")


---
## `fetch_market_data()`
Hits the `/coins/markets` endpoint. Returns one record per coin with price, 
volume, market cap, ATH, and % change over 1h / 24h / 7d.


In [ ]:
def fetch_market_data(
    vs_currency: str = "usd",
    coins: list = COINS,
    sparkline: bool = False
) -> Optional[list]:
    """
    Fetch current market data for a list of coins from CoinGecko.

    Args:
        vs_currency: Target currency for pricing (default: 'usd')
        coins: List of CoinGecko coin IDs
        sparkline: Include 7-day sparkline data (default: False)

    Returns:
        List of raw coin dicts from the API
    """
    endpoint = f"{BASE_URL}/coins/markets"
    params = {
        "vs_currency": vs_currency,
        "ids": ",".join(coins),
        "order": "market_cap_desc",
        "per_page": len(coins),
        "page": 1,
        "sparkline": str(sparkline).lower(),
        "price_change_percentage": "1h,24h,7d"
    }

    logger.info(f"Fetching market data for {len(coins)} coins...")
    response = requests.get(endpoint, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()
    logger.info(f"✓ Fetched {len(data)} coin records")
    return data


In [ ]:
# ── Run the extract ──────────────────────────────────────────────
raw_market_data = fetch_market_data()

print(f"\nRecords returned: {len(raw_market_data)}")
print(f"Keys in each record: {len(raw_market_data[0])} fields")
print(f"\nAll available fields:")
for k in raw_market_data[0].keys():
    print(f"  {k}")


In [ ]:
# ── Inspect a single record ──────────────────────────────────────
first = raw_market_data[0]
print(f"Sample record — {first['name']} ({first['symbol'].upper()})")
print("-" * 50)

interesting_fields = [
    "id", "symbol", "name", "current_price", "market_cap",
    "market_cap_rank", "total_volume", "high_24h", "low_24h",
    "price_change_percentage_24h_in_currency",
    "price_change_percentage_1h_in_currency",
    "price_change_percentage_7d_in_currency",
    "circulating_supply", "ath", "last_updated"
]

for field in interesting_fields:
    val = first.get(field)
    print(f"  {field:<50} {val}")


In [ ]:
# ── Quick price summary table ────────────────────────────────────
print(f"{'Rank':<6} {'Coin':<14} {'Symbol':<8} {'Price (USD)':>14} {'24h %':>8} {'Market Cap':>16}")
print("-" * 70)
for c in raw_market_data:
    rank = c.get("market_cap_rank", "-")
    name = c.get("name", "")[:13]
    sym  = (c.get("symbol") or "").upper()
    price = c.get("current_price") or 0
    chg   = c.get("price_change_percentage_24h_in_currency") or 0
    mcap  = c.get("market_cap") or 0
    arrow = "▲" if chg >= 0 else "▼"
    print(f"  {rank:<4} {name:<14} {sym:<8} ${price:>13,.4f} {arrow}{abs(chg):>6.2f}% ${mcap:>14,.0f}")


---
## `fetch_global_stats()`
Hits `/global`. Returns aggregated stats for the entire crypto market —
total market cap, 24h volume, BTC/ETH dominance, active coin count.


In [ ]:
def fetch_global_stats() -> Optional[dict]:
    """
    Fetch global crypto market statistics.

    Returns:
        Dict of global market stats from the /global endpoint
    """
    endpoint = f"{BASE_URL}/global"
    logger.info("Fetching global market statistics...")
    response = requests.get(endpoint, timeout=15)
    response.raise_for_status()
    data = response.json().get("data", {})
    logger.info("✓ Fetched global stats")
    return data


In [ ]:
raw_global = fetch_global_stats()

total_mcap = raw_global.get("total_market_cap", {}).get("usd", 0)
total_vol  = raw_global.get("total_volume", {}).get("usd", 0)
btc_dom    = raw_global.get("market_cap_percentage", {}).get("btc", 0)
eth_dom    = raw_global.get("market_cap_percentage", {}).get("eth", 0)
active     = raw_global.get("active_cryptocurrencies", 0)

print("Global Market Snapshot")
print("=" * 40)
print(f"  Total market cap  : ${total_mcap:>20,.0f}")
print(f"  24h volume        : ${total_vol:>20,.0f}")
print(f"  BTC dominance     : {btc_dom:>19.2f}%")
print(f"  ETH dominance     : {eth_dom:>19.2f}%")
print(f"  Active coins      : {active:>20,}")


---
## ✅ Extract complete

`raw_market_data` and `raw_global` are ready to pass into **02_transform.ipynb**.

The next notebook cleans nulls, normalizes timestamps, derives a `volatility_score`,
and shapes the data into a database-ready schema.
